### 1. Imports et chemins

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from tqdm import tqdm

# Chemin vers le dataset Kaggle
BASE_PATH = "/kaggle/input/datasets/mouhamed221/dataset-spark-week7/dataset/"

### 2. Transformations et Dataset

In [2]:
# Transformations pour train (augmentation + normalisation)
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# Transformations pour val et test (normalisation seulement)
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# Dataset PyTorch (celui que tu as déjà)
class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, base_path, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.base_path = base_path
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.base_path + self.df.iloc[idx]['filepath']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, self.df.iloc[idx]['id']
        else:
            label = self.df.iloc[idx]['target']
            return image, torch.tensor(label, dtype=torch.float32)

# Création des datasets
train_dataset = ChestXrayDataset(BASE_PATH + "train.csv", BASE_PATH, transform=train_transform)
val_dataset = ChestXrayDataset(BASE_PATH + "val.csv", BASE_PATH, transform=val_transform)
test_dataset = ChestXrayDataset(BASE_PATH + "test.csv", BASE_PATH, transform=val_transform, is_test=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_pos = train_dataset.df['target'].sum()
num_neg = len(train_dataset.df) - num_pos

pos_weight = torch.tensor([num_neg / num_pos]).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 2. Modele CNN

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)


### 3. Fonction de perte et l'optimiseur

In [4]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

### 4. Boucle d’entraînement avec validation

In [5]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device).float()

        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_preds, val_labels, val_probs = [], [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()
            probs = torch.sigmoid(outputs)

            val_probs.extend(probs.cpu().tolist())
            val_preds.extend((probs > 0.5).cpu().int().tolist())
            val_labels.extend(labels.cpu().int().tolist())

    # Métriques
    accuracy = (torch.tensor(val_preds) == torch.tensor(val_labels)).float().mean()
    f1 = f1_score(val_labels, val_preds)

    try:
        auc = roc_auc_score(val_labels, val_probs)
    except:
        auc = 0.0

    cm = confusion_matrix(val_labels, val_preds)

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f} | Acc: {accuracy:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")

100%|██████████| 129/129 [08:06<00:00,  3.77s/it]


Epoch 1/10 - Loss: 0.6262 | Acc: 0.9191 | F1: 0.9429 | AUC: 0.9830
Confusion Matrix:
[[221  16]
 [ 55 586]]


100%|██████████| 129/129 [07:16<00:00,  3.38s/it]


Epoch 2/10 - Loss: 0.1426 | Acc: 0.9294 | F1: 0.9504 | AUC: 0.9851
Confusion Matrix:
[[222  15]
 [ 47 594]]


100%|██████████| 129/129 [07:21<00:00,  3.42s/it]


...

Epoch 8/10 - Loss: 0.1123 | Acc: 0.9248 | F1: 0.9467 | AUC: 0.9868
Confusion Matrix:
[[226  11]
 [ 55 586]]


100%|██████████| 129/129 [07:31<00:00,  3.50s/it]


Epoch 9/10 - Loss: 0.1149 | Acc: 0.9021 | F1: 0.9288 | AUC: 0.9877
Confusion Matrix:
[[231   6]
 [ 80 561]]


100%|██████████| 129/129 [07:29<00:00,  3.49s/it]


Epoch 10/10 - Loss: 0.1048 | Acc: 0.9465 | F1: 0.9631 | AUC: 0.9889
Confusion Matrix:
[[217  20]
 [ 27 614]]


### 5. Prédiction sur test et génération de submission.csv

In [6]:
model.eval()
predictions, ids = [], []

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()
        predictions.extend(preds.cpu().tolist())
        ids.extend(img_ids)

submission = pd.DataFrame({'id': ids, 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

In [7]:
import os
print(os.listdir('/kaggle/working/'))

['submission.csv', '.virtual_documents']
